# Import Required Libraries
Import the necessary Python packages for PDF handling, data manipulation, and plotting.

In [ ]:
import re
import json
from datetime import datetime

# PDF parsing
import PyPDF2

# data and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ensure inline plots
%matplotlib inline

# Load Sample PDF
Read the provided sample timetable PDF using PyPDF2 and prepare it for text extraction.

In [ ]:
sample_path = r"d:\\Documents from D\\INDIAN RAILWAYS TRAINS TIME TABLE\\IRI-12084-Coimbatore - Mayiladuthurai Jan Shatabdi Express.pdf"
reader = PyPDF2.PdfReader(sample_path)
print(f"Loaded {len(reader.pages)} pages from sample PDF")

# Extract Text from PDF
Iterate over each page, extract the raw text, and concatenate it for parsing.

In [ ]:
raw_pages = []
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_pages.append(text)

raw_text = "
".join(raw_pages)
print(raw_text[:1000])  # show a snippet

# Parse Train Timetable
Use regular expressions and string manipulation to pull out useful fields such as train number, name, origin/destination and station schedule rows.

In [ ]:
def parse_timetable(text: str):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    result = {}

    # train number / name
    if lines:
        first = lines[0]
        m = re.match(r"(\d+)\s*/\s*(.+)", first)
        if m:
            result['number'] = m.group(1)
            result['name'] = m.group(2)
        else:
            parts = first.split(None,1)
            result['number'] = parts[0]
            result['name'] = parts[1] if len(parts)>1 else ''

    # days of service
    days = []
    for l in lines:
        if 'Departs' in l or 'Runs' in l:
            m = re.search(r'Departs\s+([A-Za-z,]+)', l)
            if m:
                names = [d.strip() for d in m.group(1).split(',')]
                mapping = {'Sun':0,'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6}
                days = [mapping[d] for d in names if d in mapping]
    result['daysOfService'] = days

    # schedule rows
    schedule = []
    header_idx = None
    for idx,l in enumerate(lines):
        if l.startswith('#Code'):
            header_idx = idx
            break
    if header_idx is not None:
        for row in lines[header_idx+1:]:
            parts = re.split(r\
, row)
            if len(parts) >= 4:
                code,name,arr,dep = parts[0], parts[1], parts[2], parts[3]
                schedule.append({'code':code,'name':name,'arrival':arr,'departure':dep})
    result['schedule'] = schedule
    return result

parsed = parse_timetable(raw_text)
parsed

# Create DataFrame of Schedule
Convert the parsed schedule list into a pandas DataFrame for easy examination.

In [ ]:
df = pd.DataFrame(parsed.get('schedule', []))
df[['arrival','departure']] = df[['arrival','departure']].replace({'--': None})
df.head()

# Visualize Route and Times
Plot the train's progression through stations versus time or index to get a sense of the route.

In [ ]:
# convert times to datetime for plotting
for col in ['arrival','departure']:
    df[col] = pd.to_datetime(df[col], format='%H:%M', errors='coerce')

plt.figure(figsize=(8,4))
plt.plot(df.index, df['departure'], marker='o')
plt.xticks(df.index, df['code'], rotation=45)
plt.ylabel('Departure time')
plt.title(f"Schedule for {parsed.get('number')} - {parsed.get('name')}")
plt.tight_layout();

# Define Functions for Reuse
Encapsulate the loading, extraction and parsing logic so it can be applied to another PDF easily.

In [ ]:
def extract_text_from_pdf(path: str) -> str:
    reader = PyPDF2.PdfReader(path)
    texts = [p.extract_text() or '' for p in reader.pages]
    return "
".join(texts)

def schedule_from_pdf(path: str) -> pd.DataFrame:
    raw = extract_text_from_pdf(path)
    info = parse_timetable(raw)
    return pd.DataFrame(info.get('schedule', []))

# test helper on sample
df2 = schedule_from_pdf(sample_path)
df2.head()

# Apply Analysis to New Train PDF
Load a different timetable PDF (replace the path) and run the same routines to demonstrate reuse.

In [ ]:
# example path for a second PDF -- update when available
new_pdf = r"d:\\path\\to\\another\\train.pdf"  # placeholder

# if the file exists, process it:
import os
if os.path.exists(new_pdf):
    df_new = schedule_from_pdf(new_pdf)
    display(df_new.head())
else:
    print('New train PDF not found, update the path and rerun.')